In [7]:
import math
from datetime import datetime, timedelta
from typing import Optional
import xml.etree.ElementTree as ET

def meters_per_deg_lat(lat_deg: float) -> float:
    """Meters per degree of latitude (WGS84 approximation)."""
    phi = math.radians(lat_deg)
    return (
        111132.92
        - 559.82 * math.cos(2 * phi)
        + 1.175 * math.cos(4 * phi)
        - 0.0023 * math.cos(6 * phi)
    )

def meters_per_deg_lon(lat_deg: float) -> float:
    """Meters per degree of longitude (WGS84 approximation)."""
    phi = math.radians(lat_deg)
    return (
        111412.84 * math.cos(phi)
        - 93.5 * math.cos(3 * phi)
        + 0.118 * math.cos(5 * phi)
    )

def generate_square_gpx_fast_vertical(
    start_lat: float = 22.198745,   # Default: Macau approx center
    start_lon: float = 113.543873,
    west_m: float = 0.0,
    north_m: float = 0.0,
    east_m: float = 100.0,
    south_m: float = 100.0,
    horizontal_step_m: float = 1.0,   # eastward step between columns
    secs_per_meter: float = 1.0,      # time pacing (s per meter)
    filename: str = "square_fast_vertical.gpx",
    start_time_utc: Optional[datetime] = None,  # <-- Python 3.8 compatible
    max_columns: int = 100_000,       # safety cap on number of columns
    allow_exceed_east: bool = True    # last east step can slightly exceed east boundary
):
    """
    Generate a GPX track that sweeps a rectangle:
      - Corner UL = start shifted west and north by (west_m, north_m)
      - Corner LR = start shifted east and south by (east_m, south_m)

    Pattern:
      1) Start at UL
      2) One big vertical move to the opposite north/south boundary
      3) Step east by ~horizontal_step_m
      4) One big vertical move back
      5) Repeat until east boundary is reached (or slightly exceeded if allowed)
    """
    # Validate inputs
    for name, v in {
        "west_m": west_m, "north_m": north_m, "east_m": east_m, "south_m": south_m,
        "horizontal_step_m": horizontal_step_m, "secs_per_meter": secs_per_meter
    }.items():
        if v < 0:
            raise ValueError(f"{name} must be non-negative.")

    # Compute UL and LR corners (use local scale at start latitude; accurate for local rectangles)
    lat_ul = start_lat + (north_m / meters_per_deg_lat(start_lat))
    lon_ul = start_lon - (west_m / meters_per_deg_lon(start_lat))
    lat_lr = start_lat - (south_m / meters_per_deg_lat(start_lat))
    lon_lr = start_lon + (east_m / meters_per_deg_lon(start_lat))

    if not (lat_ul > lat_lr):
        raise ValueError("Computed UL must be north of LR. Check north_m/south_m.")
    if not (lon_ul < lon_lr):
        raise ValueError("Computed UL must be west of LR. Check west_m/east_m.")

    # Geometry estimates (at mid-lat for better scale)
    mid_lat = (lat_ul + lat_lr) / 2.0
    width_m_est = (lon_lr - lon_ul) * meters_per_deg_lon(mid_lat)
    height_m_est = (lat_ul - lat_lr) * meters_per_deg_lat(mid_lat)

    # Columns estimate and safety
    est_cols = max(1, int(math.ceil(width_m_est / max(1e-9, horizontal_step_m))))
    if est_cols > max_columns:
        raise RuntimeError(
            f"Estimated columns ~{est_cols:,} (> max_columns={max_columns:,}). "
            f"Increase horizontal_step_m or raise max_columns."
        )

    # Prepare GPX
    if start_time_utc is None:
        start_time_utc = datetime.utcnow()
    gpx = ET.Element(
        "gpx", 
        version="1.1",
        creator="SquareGPXGeneratorFastVertical",
        xmlns="http://www.topografix.com/GPX/1/1"
    )
    trk = ET.SubElement(gpx, "trk")
    ET.SubElement(trk, "name").text = "Square coverage pattern (giant vertical steps)"
    trkseg = ET.SubElement(trk, "trkseg")

    def add_trkpt(lat: float, lon: float, t: datetime):
        trkpt = ET.SubElement(trkseg, "trkpt", lat=f"{lat:.7f}", lon=f"{lon:.7f}")
        ET.SubElement(trkpt, "ele").text = "0"
        ET.SubElement(trkpt, "time").text = t.strftime("%Y-%m-%dT%H:%M:%SZ")

    current_time = start_time_utc
    cur_lat, cur_lon = lat_ul, lon_ul
    add_trkpt(cur_lat, cur_lon, current_time)

    total_m = 0.0
    going_south = True

    def move_vertical(to_lat: float):
        """Single giant vertical segment to 'to_lat' at current longitude."""
        nonlocal cur_lat, cur_lon, current_time, total_m
        if cur_lat == to_lat:
            return
        avg_lat = (cur_lat + to_lat) / 2.0
        meters = abs(to_lat - cur_lat) * meters_per_deg_lat(avg_lat)
        total_m += meters
        current_time += timedelta(seconds=meters * secs_per_meter)
        cur_lat = to_lat
        add_trkpt(cur_lat, cur_lon, current_time)

    def step_east(allow_exceed: bool):
        """Step east by ~horizontal_step_m (single point)."""
        nonlocal cur_lat, cur_lon, current_time, total_m
        if horizontal_step_m == 0:
            return
        m_per_deg_lon_here = meters_per_deg_lon(cur_lat)
        desired_step_deg = horizontal_step_m / m_per_deg_lon_here
        next_lon = cur_lon + desired_step_deg
        if not allow_exceed and next_lon > lon_lr:
            next_lon = lon_lr
        meters = abs(next_lon - cur_lon) * meters_per_deg_lon(cur_lat)
        total_m += meters
        current_time += timedelta(seconds=meters * secs_per_meter)
        cur_lon = next_lon
        add_trkpt(cur_lat, cur_lon, current_time)

    # Build the pattern
    while True:
        # 1) Giant vertical move to the relevant boundary
        target_lat = lat_lr if going_south else lat_ul
        move_vertical(target_lat)

        # 2) If already at or beyond east boundary, stop
        if cur_lon >= lon_lr:
            break

        # 3) East step (possibly the last one)
        m_per_deg_lon_here = meters_per_deg_lon(cur_lat)
        would_next_lon = cur_lon + (horizontal_step_m / m_per_deg_lon_here)
        if would_next_lon > lon_lr:
            step_east(allow_exceed=allow_exceed_east)
            break
        else:
            step_east(allow_exceed=True)
            # 4) Toggle vertical direction for next column
            going_south = not going_south

    # Save GPX
    ET.ElementTree(gpx).write(filename, encoding="utf-8", xml_declaration=True)

    summary = {
        "file": filename,
        "ul_corner": (lat_ul, lon_ul),
        "lr_corner": (lat_lr, lon_lr),
        "approx_width_m": width_m_est,
        "approx_height_m": height_m_est,
        "columns_est": est_cols,
        "track_points": len(list(trkseg)),
        "distance_m": total_m
    }
    print(
        f"Saved GPX to {filename}\n"
        f" UL: ({lat_ul:.6f}, {lon_ul:.6f})\n"
        f" LR: ({lat_lr:.6f}, {lon_lr:.6f})\n"
        f" Width ≈ {width_m_est:.2f} m, Height ≈ {height_m_est:.2f} m\n"
        f" Columns ≈ {est_cols:,}\n"
        f" Track points: {summary['track_points']:,}\n"
        f" Total path ≈ {total_m:.1f} m"
    )
    return summary

In [8]:
#macau

#start_lat=22.198745,   # Macau latitude
#start_lon=113.543873,  # Macau longitude

#VAT 41°54′10″N，12°27′11″E）
#start_lat=41.902778,   # VAT latitude
#start_lon=12.453056,  # VAT longitude

generate_square_gpx_fast_vertical(
    start_lat=41.902778,   
    start_lon=12.453056,
    #west_m=2500, north_m=3000, # MAC
    #east_m=5400, south_m=10500, # MAC
    west_m=2000, north_m=2000, # VAT
    east_m=2000, south_m=2000, # VAT
    horizontal_step_m=3.0,
    #filename="MAC_square_solid.gpx",
    filename="VAT_square_solid.gpx"
)

Saved GPX to VAT_square_solid.gpx
 UL: (41.920784, 12.428953)
 LR: (41.884772, 12.477159)
 Width ≈ 4000.00 m, Height ≈ 4000.00 m
 Columns ≈ 1,334
 Track points: 2,669
 Total path ≈ 5340002.0 m


{'file': 'VAT_square_solid.gpx',
 'ul_corner': (41.920784440871984, 12.428952855419613),
 'lr_corner': (41.88477155912801, 12.477159144580387),
 'approx_width_m': 3999.99999999994,
 'approx_height_m': 4000.0000000003297,
 'columns_est': 1334,
 'track_points': 2669,
 'distance_m': 5340002.00000039}